# Limpieza de Textos - Dataset Zenodo

**Objetivo:** Limpiar los textos eliminando menciones, URLs, hashtags y caracteres innecesarios

**Por qué es importante:**
- Las menciones (@usuario) no aportan al modelo
- Las URLs no tienen significado lingüístico
- Los hashtags a veces distorsionan el mensaje
- Los emojis pueden confundir el modelo

**Dataset:** zenodo_limpio.csv (5,935 ejemplos)

---

In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Librerias importadas correctamente")

Librerias importadas correctamente


In [11]:
# Cargar dataset limpio de Fase 1
df = pd.read_csv('../data/processed/zenodo_limpio.csv')

print("="*70)
print("DATASET ORIGINAL")
print("="*70)

print("\nTotal:", len(df), "ejemplos")

print("\nDistribucion por clase:")
print(df['label'].value_counts())

# Mostrar ejemplos con menciones
print("\n" + "="*70)
print("EJEMPLOS CON MENCIONES (@)")
print("="*70)

con_menciones = df[df['texto'].str.contains('@', na=False)].head(5)
print(f"\nTotal con menciones: {df['texto'].str.contains('@', na=False).sum()}")
print("\nPrimeros 5 ejemplos:")
for i, texto in enumerate(con_menciones['texto'], 1):
    print(f"   {i}. {texto[:100]}...")

# Mostrar ejemplos con URLs
print("\n" + "="*70)
print("EJEMPLOS CON URLS")
print("="*70)

con_urls = df[df['texto'].str.contains('http', na=False)].head(5)
print(f"\nTotal con URLs: {df['texto'].str.contains('http', na=False).sum()}")
print("\nPrimeros 5 ejemplos:")
for i, texto in enumerate(con_urls['texto'], 1):
    print(f"   {i}. {texto[:100]}...")

# Mostrar ejemplos con hashtags
print("\n" + "="*70)
print("EJEMPLOS CON HASHTAGS (#)")
print("="*70)

con_hashtags = df[df['texto'].str.contains('#', na=False)].head(5)
print(f"\nTotal con hashtags: {df['texto'].str.contains('#', na=False).sum()}")
print("\nPrimeros 5 ejemplos:")
for i, texto in enumerate(con_hashtags['texto'], 1):
    print(f"   {i}. {texto[:100]}...")

DATASET ORIGINAL

Total: 5935 ejemplos

Distribucion por clase:
label
0    4368
1    1567
Name: count, dtype: int64

EJEMPLOS CON MENCIONES (@)

Total con menciones: 2832

Primeros 5 ejemplos:
   1. @SuperFalete jajajajaja. Te jodes maricon....
   2. @Barandaand @lapaseata Los actores estos es que son todos unos fachas ......
   3. @MrAlvatros25 @Edd33Martin ha hecho las dos subnormal...
   4. @SabakJacare Ideas racistas, xenófobas, machistas... eso es lo que defiende Trump. Y lo siento pero ...
   5. @Francis76440634 El Manchego Maricón debería tener esa sucia boca callada. Lo digo por su bien. Lueg...

EJEMPLOS CON URLS

Total con URLs: 1733

Primeros 5 ejemplos:
   1. Muy despreciados,siiii,pero todos vestidos de alta costura española o extranjera como la derechona b...
   2. marica explicame porque a veces no te entiendo. — En vez de venir a ensuciar mi cc con insultos homo...
   3. Abusan de 1400 niñas durante 16 años porque la policía tenía miedo de que les llamasen racistas, era

In [12]:
def limpiar_texto(texto):
    """
    Limpia un texto eliminando menciones, URLs, hashtags, emojis y normalizando
    
    Parametros:
        texto: String con el texto a limpiar
    
    Retorna:
        String con el texto limpio
    """
    
    # Convertir a string por si acaso
    texto = str(texto)
    
    # 1. Eliminar URLs
    texto = re.sub(r'http\S+|www\.\S+', '', texto)
    
    # 2. Eliminar menciones
    texto = re.sub(r'@\w+', '', texto)
    
    # 3. Eliminar @ seguido de espacios o caracteres especiales
    texto = re.sub(r'@\s+', '', texto)
    texto = re.sub(r'@[^\w\s]+', '', texto)
    
    # 4. Eliminar hashtags (quitar # pero dejar palabra)
    texto = re.sub(r'#(\w+)', r'\1', texto)
    
    # 5. Eliminar RT (retweet)
    texto = re.sub(r'\bRT\b', '', texto, flags=re.IGNORECASE)
    
    # 6. Eliminar emojis y caracteres especiales
    # Mantener solo: letras, numeros, espacios, puntuacion basica, acentos
    texto = re.sub(r'[^\w\s,.!?¿¡;:\-()áéíóúñÁÉÍÓÚÑüÜ]', '', texto)
    
    # 7. Eliminar saltos de linea y tabulaciones
    texto = texto.replace('\n', ' ').replace('\t', ' ').replace('\r', ' ')
    
    # 8. Eliminar espacios multiples
    texto = re.sub(r'\s+', ' ', texto)
    
    # 9. Eliminar espacios al inicio y final
    texto = texto.strip()
    
    # 10. Eliminar puntos suspensivos al final
    texto = re.sub(r'\.\.\.+$', '', texto)
    
    return texto

# Probar con ejemplos reales
ejemplos_prueba = [
    "😎😎🌴🏖️ @ Playa Sucia, Cabo Rojo, PR",
    "Pasitoo a pasitoo, suave suavecitoo...‼️🍻💃🏻🔙🔥 @ Las Machorras",
    "Parezco monguer @ Jadraque, Spain",
    "@SuperFalete jajajajaja. Te jodes maricon....",
    "Mira este link http://example.com que interesante #bullying"
]

print("="*70)
print("PRUEBAS DE LIMPIEZA")
print("="*70)

for i, ejemplo in enumerate(ejemplos_prueba, 1):
    limpio = limpiar_texto(ejemplo)
    print(f"\n{i}. ANTES:")
    print(f"   {ejemplo}")
    print(f"   DESPUES:")
    print(f"   {limpio}")

print("\n" + "="*70)
print("Funcion de limpieza definida y probada")

PRUEBAS DE LIMPIEZA

1. ANTES:
   😎😎🌴🏖️ @ Playa Sucia, Cabo Rojo, PR
   DESPUES:
   Playa Sucia, Cabo Rojo, PR

2. ANTES:
   Pasitoo a pasitoo, suave suavecitoo...‼️🍻💃🏻🔙🔥 @ Las Machorras
   DESPUES:
   Pasitoo a pasitoo, suave suavecitoo... Las Machorras

3. ANTES:
   Parezco monguer @ Jadraque, Spain
   DESPUES:
   Parezco monguer Jadraque, Spain

4. ANTES:
   @SuperFalete jajajajaja. Te jodes maricon....
   DESPUES:
   jajajajaja. Te jodes maricon

5. ANTES:
   Mira este link http://example.com que interesante #bullying
   DESPUES:
   Mira este link que interesante bullying

Funcion de limpieza definida y probada


In [13]:
print("="*70)
print("APLICANDO LIMPIEZA A TODO EL DATASET")
print("="*70)

# Crear copia para no modificar el original
df_limpio = df.copy()

# Guardar textos originales para comparacion
df_limpio['texto_original'] = df_limpio['texto']

# Aplicar limpieza a todos los textos
print("\nLimpiando", len(df_limpio), "textos...")
df_limpio['texto'] = df_limpio['texto'].apply(limpiar_texto)

print("Limpieza aplicada")

# Eliminar textos que quedaron vacios despues de limpiar
antes = len(df_limpio)
df_limpio = df_limpio[df_limpio['texto'].str.len() > 0]
despues = len(df_limpio)

if antes - despues > 0:
    print(f"\nTextos vacios eliminados: {antes - despues}")
else:
    print("\nNo se encontraron textos vacios")

print(f"Total despues de eliminar vacios: {despues}")

# Eliminar textos muy cortos (menos de 15 caracteres)
antes = len(df_limpio)
df_limpio = df_limpio[df_limpio['texto'].str.len() >= 15]
despues = len(df_limpio)

if antes - despues > 0:
    print(f"Textos muy cortos (<15 chars) eliminados: {antes - despues}")
else:
    print("No se encontraron textos muy cortos")

print(f"Total final: {despues}")

print("\n" + "="*70)
print("LIMPIEZA COMPLETADA")
print("="*70)

APLICANDO LIMPIEZA A TODO EL DATASET

Limpiando 5935 textos...
Limpieza aplicada

No se encontraron textos vacios
Total despues de eliminar vacios: 5935
Textos muy cortos (<15 chars) eliminados: 4
Total final: 5931

LIMPIEZA COMPLETADA


In [14]:
print("="*70)
print("COMPARACION ANTES Y DESPUES")
print("="*70)

# Mostrar ejemplos comparando antes/despues
print("\nEjemplos de limpieza (primeros 15):")
print("-"*70)

contador = 0
for i in range(len(df_limpio)):
    if contador >= 15:
        break
    
    original = df_limpio.iloc[i]['texto_original']
    limpio = df_limpio.iloc[i]['texto']
    
    # Solo mostrar si hubo cambios
    if original != limpio:
        contador += 1
        print(f"\n{contador}. ANTES:")
        print(f"   {original[:100]}...")
        print(f"   DESPUES:")
        print(f"   {limpio[:100]}...")

# Estadisticas de limpieza
print("\n" + "="*70)
print("ESTADISTICAS DE LIMPIEZA")
print("="*70)

# Contar menciones
menciones_antes = df['texto'].str.contains('@', na=False).sum()
menciones_despues = df_limpio['texto'].str.contains('@', na=False).sum()

print(f"\nMenciones (@):")
print(f"   Antes: {menciones_antes} textos")
print(f"   Despues: {menciones_despues} textos")
print(f"   Eliminadas: {menciones_antes - menciones_despues}")

# Contar URLs
urls_antes = df['texto'].str.contains('http', na=False).sum()
urls_despues = df_limpio['texto'].str.contains('http', na=False).sum()

print(f"\nURLs (http):")
print(f"   Antes: {urls_antes} textos")
print(f"   Despues: {urls_despues} textos")
print(f"   Eliminadas: {urls_antes - urls_despues}")

# Contar hashtags
hashtags_antes = df['texto'].str.contains('#', na=False).sum()
hashtags_despues = df_limpio['texto'].str.contains('#', na=False).sum()

print(f"\nHashtags (#):")
print(f"   Antes: {hashtags_antes} textos")
print(f"   Despues: {hashtags_despues} textos")
print(f"   Eliminados: {hashtags_antes - hashtags_despues}")

# Longitud promedio
long_antes = df['texto'].str.len().mean()
long_despues = df_limpio['texto'].str.len().mean()

print(f"\nLongitud promedio:")
print(f"   Antes: {long_antes:.1f} caracteres")
print(f"   Despues: {long_despues:.1f} caracteres")
print(f"   Diferencia: {long_antes - long_despues:.1f} caracteres")

COMPARACION ANTES Y DESPUES

Ejemplos de limpieza (primeros 15):
----------------------------------------------------------------------

1. ANTES:
   Ismael es egocentrico porque se vuelve loca si le dicen que tiene el pelo bonito😂😂😂😂 eso se define c...
   DESPUES:
   Ismael es egocentrico porque se vuelve loca si le dicen que tiene el pelo bonito eso se define con o...

2. ANTES:
   ..ya tardaba en salir quien pronunciase nombre catalán sílaba aguda como si fuese plana [es Eduááárd...
   DESPUES:
   ..ya tardaba en salir quien pronunciase nombre catalán sílaba aguda como si fuese plana es Eduááárd,...

3. ANTES:
   (Esto no es un discurso político y razonado, obviamente, solo una llamada de atención en plan "JODER...
   DESPUES:
   (Esto no es un discurso político y razonado, obviamente, solo una llamada de atención en plan JODER,...

4. ANTES:
   Muy despreciados,siiii,pero todos vestidos de alta costura española o extranjera como la derechona b...
   DESPUES:
   Muy despreciados,sii

In [15]:
import string

def tiene_caracteres_raros(texto):
    """
    Verifica si un texto tiene caracteres raros (emojis, simbolos especiales)
    """
    # Caracteres permitidos
    permitidos = set(string.ascii_letters + string.digits + string.punctuation + 
                     string.whitespace + 'áéíóúñÁÉÍÓÚÑüÜ')
    
    for char in texto:
        if char not in permitidos:
            return True
    return False

# Verificar caracteres raros
con_raros = df_limpio['texto'].apply(tiene_caracteres_raros).sum()

print("="*70)
print("VERIFICACION FINAL - CARACTERES RAROS")
print("="*70)

print(f"\nTextos con caracteres raros (emojis, simbolos): {con_raros}")
print(f"Total textos: {len(df_limpio)}")
print(f"Porcentaje: {(con_raros/len(df_limpio)*100):.2f}%")

if con_raros == 0:
    print("\n DATASET COMPLETAMENTE LIMPIO")
else:
    print(f"\nQuedan {con_raros} textos con caracteres raros")
    print("\nPrimeros 5 ejemplos:")
    raros = df_limpio[df_limpio['texto'].apply(tiene_caracteres_raros)].head(5)
    for i, texto in enumerate(raros['texto'], 1):
        print(f"{i}. {texto}")

VERIFICACION FINAL - CARACTERES RAROS

Textos con caracteres raros (emojis, simbolos): 409
Total textos: 5931
Porcentaje: 6.90%

Quedan 409 textos con caracteres raros

Primeros 5 ejemplos:
1. ¿eres heterofobica o no?, no te entiendo amiguita. La heterofobia no existe cariño
2. Basta ya de levantar fronteras, dice el premiado al Mejor Documental. ¿Lo dirá en contra del secesionismo catalán o en contra de Trump?
3. Quede ofendida marica, ¿como va a decir eso?
4. ¿y eso es ser racista?Eso es ser pobre pedazo subnormal. Y el dinero se lo lleva Morales haciendo un museo de 8 millones ahí.
5. ¿Esa es tu idea de patriotismo? Las empresas separatistas tb son minoría en la Cataluña, como los abducidos por MafiaPujol


In [ ]:
# Eliminar columna temporal
df_final = df_limpio[['texto', 'label', 'fuente']].copy()

# Guardar
output_path = '../data/processed/zenodo_limpio_textos.csv'
df_final.to_csv(output_path, index=False, encoding='utf-8')

print("="*70)
print("DATASET LIMPIO GUARDADO")
print("="*70)

print(f"\nUbicacion: {output_path}")
print(f"Total: {len(df_final)} ejemplos")

print("\nDistribucion final:")
distribucion = df_final['label'].value_counts().sort_index()
for label, count in distribucion.items():
    pct = (count / len(df_final)) * 100
    nombre = "No ofensivo" if label == 0 else "Ofensivo"
    print(f"   Clase {label} ({nombre}): {count} ({pct:.1f}%)")

print("\nEjemplos finales (primeros 10 textos limpios):")
for i, texto in enumerate(df_final['texto'].head(10), 1):
    print(f"   {i}. {texto[:80]}...")

print("\n" + "="*70)
print(" LIMPIEZA DE TEXTOS COMPLETADA")
print("="*70)


DATASET LIMPIO GUARDADO

Ubicacion: ../data/processed/zenodo_limpio_textos.csv
Total: 5931 ejemplos

Distribucion final:
   Clase 0 (No ofensivo): 4366 (73.6%)
   Clase 1 (Ofensivo): 1565 (26.4%)

Ejemplos finales (primeros 10 textos limpios):
   1. Ismael es egocentrico porque se vuelve loca si le dicen que tiene el pelo bonito...
   2. ..ya tardaba en salir quien pronunciase nombre catalán sílaba aguda como si fues...
   3. (Esto no es un discurso político y razonado, obviamente, solo una llamada de ate...
   4. Muy despreciados,siiii,pero todos vestidos de alta costura española o extranjera...
   5. marica explicame porque a veces no te entiendo. En vez de venir a ensuciar mi cc...
   6. Abusan de 1400 niñas durante 16 años porque la policía tenía miedo de que les ll...
   7. Es ridículo, ayer le dije a un árabe que no era necesario comer culos que no iba...
   8. 39. Me considero género no binario y mis pronombres son ella elle...
   9. jajajajaja. Te jodes maricon....
   10. Pero 